In [1]:
%%writefile /shared/ogi_navi_sad/cd_state_updater.py
from __future__ import annotations
import numpy as np
import torch
import torch.nn as nn
from dataclasses import dataclass
from typing import Literal, List
import warnings

try:
    from cd import solve_1d_picard, viability_threshold_1d, check_convergence, solution_type
    CD_AVAILABLE = True
except ImportError:
    CD_AVAILABLE = False

D_STATE = 64
D_MSG   = 64
D_QUERY = 32

@dataclass
class CDUpdateRecord:
    step: int; module_id: str; converged: bool
    solution_type: Literal['trivial','nontrivial']
    max_phi: float; beta_used: float; beta_star: float
    above_threshold: bool; iterations: int
    @property
    def bifurcated(self): return self.solution_type == 'nontrivial'

class CDStateUpdater:
    def __init__(self, d_state=D_STATE, L=1.0, c=10.0, beta_scale=1.2):
        self.d_state=d_state; self.N=d_state; self.L=L; self.c=c
        self.beta_scale=beta_scale; self.records=[]
        if CD_AVAILABLE:
            self._beta_star_unit = viability_threshold_1d(L, b=1.0)

    def _message_to_viability(self, msg):
        m = msg.flatten()[:self.N]
        lo,hi = m.min(),m.max()
        b = np.ones(self.N)*0.5 if hi-lo<1e-8 else (m-lo)/(hi-lo)
        return b, float(b.mean())

    def _query_to_drive(self, q):
        q = q.flatten()[:D_QUERY]
        a = np.interp(np.linspace(0,1,self.N), np.linspace(0,1,len(q)), q)
        return np.abs(a)

    def update(self, compressed, state, goal_query, step=0, module_id='?'):
        msg_np   = compressed.detach().numpy().flatten()
        query_np = goal_query.detach().numpy().flatten()
        b_field, b_mean = self._message_to_viability(msg_np)
        a_field = self._query_to_drive(query_np)
        beta_star = viability_threshold_1d(self.L, b=b_mean) if b_mean>1e-8 else float(self._beta_star_unit)
        beta_b = self.beta_scale * beta_star
        try:
            x, Phi, info = solve_1d_picard(L=self.L, N=self.N, a=float(a_field.mean()), beta_b=beta_b, c=self.c)
            converged = check_convergence(info)[0]
            sol_type  = solution_type(info)
            max_phi   = float(info.get('maxPhi', np.abs(Phi).max()))
            n_iter    = int(info.get('iterations', -1))
            # Always clamp to exactly N points
            Phi = Phi.flatten()[:self.N] if len(Phi) > self.N else Phi.flatten()
            Phi = Phi[:self.N]  # hard clamp
            state_norm = float(torch.norm(state).item())
            phi_norm   = float(np.linalg.norm(Phi))
            if phi_norm > 1e-8 and state_norm > 1e-8:
                Phi = Phi * (state_norm / phi_norm)
            new_state = torch.tensor(Phi, dtype=torch.float32).unsqueeze(0)
        except Exception as e:
            warnings.warn(f"CD solver failed: {e}. Using current state.")
            new_state = state; converged=False; sol_type='trivial'; max_phi=0.0; beta_star=0.0; n_iter=0
        self.records.append(CDUpdateRecord(step=step, module_id=module_id, converged=converged,
            solution_type=sol_type, max_phi=max_phi, beta_used=beta_b,
            beta_star=beta_star, above_threshold=beta_b>beta_star, iterations=n_iter))
        return new_state

    def summary(self):
        if not self.records: return {}
        n = len(self.records)
        return {'total_updates':n, 'converged':sum(1 for r in self.records if r.converged),
                'convergence_rate':sum(1 for r in self.records if r.converged)/n,
                'nontrivial':sum(1 for r in self.records if r.bifurcated),
                'bifurcation_rate':sum(1 for r in self.records if r.bifurcated)/n,
                'mean_max_phi':np.mean([r.max_phi for r in self.records])}

class FabricStateUpdater(nn.Module):
    def __init__(self, d_msg=D_MSG, d_state=D_STATE, d_query=D_QUERY, use_cd=False, beta_scale=1.2):
        super().__init__()
        self.use_cd=use_cd; self.d_state=d_state
        self.gru = nn.GRUCell(d_msg, d_state)
        self.cd  = CDStateUpdater(d_state=d_state, beta_scale=beta_scale) if CD_AVAILABLE else None
        if use_cd and not CD_AVAILABLE:
            warnings.warn("use_cd=True but CD not installed. Falling back to GRU.")
            self.use_cd=False

    def forward(self, compressed, state, goal_query, step=0, module_id='?'):
        if self.use_cd and self.cd is not None:
            return self.cd.update(compressed, state, goal_query, step, module_id)
        return self.gru(compressed, state)

    def cd_summary(self): return self.cd.summary() if self.cd else {}
    def switch_to_cd(self):
        if not CD_AVAILABLE: raise RuntimeError("CD not installed.")
        self.use_cd=True
    def switch_to_gru(self): self.use_cd=False

Writing /shared/ogi_navi_sad/cd_state_updater.py


FileNotFoundError: [Errno 2] No such file or directory: '/shared/ogi_navi_sad/cd_state_updater.py'

In [2]:
import os
print(os.getcwd())
os.listdir('.')

/mnt/custom-file-systems/s3/shared/ogi_navi_sad


['.ipynb_checkpoints',
 'Untitled.ipynb',
 'Untitled1.ipynb',
 'Untitled2.ipynb',
 'Untitled3.ipynb',
 'Untitled4.ipynb',
 'Untitled5.ipynb',
 'Untitled6.ipynb',
 '__pycache__',
 'cd_state_updater.ipynb',
 'cd_state_updater.py',
 'data',
 'fabric_attractor_geometry.png',
 'fabric_gate_selectivity.png',
 'fabric_sad_wire.png',
 'ogi_executive_gater.py',
 'ogi_executive_gater_seed123.png',
 'ogi_executive_gater_seed42.png',
 'ogi_executive_gater_seed777.png',
 'ogi_fabric_standalone.ipynb',
 'ogi_navisad_export.json',
 'ogi_navisadv3.ipynb',
 'ogi_sad_integrated.py',
 'ogi_sad_integrated_seed123.png',
 'ogi_sad_integrated_seed42.png',
 'ogi_sad_integrated_seed777.png',
 'ordinal.py',
 'pe_features.py']

In [3]:
%%writefile /mnt/custom-file-systems/s3/shared/ogi_navi_sad/cd_state_updater.py
from __future__ import annotations
import numpy as np
import torch
import torch.nn as nn
from dataclasses import dataclass
from typing import Literal, List
import warnings

try:
    from cd import solve_1d_picard, viability_threshold_1d, check_convergence, solution_type
    CD_AVAILABLE = True
except ImportError:
    CD_AVAILABLE = False

D_STATE = 64
D_MSG   = 64
D_QUERY = 32

@dataclass
class CDUpdateRecord:
    step: int; module_id: str; converged: bool
    solution_type: Literal['trivial','nontrivial']
    max_phi: float; beta_used: float; beta_star: float
    above_threshold: bool; iterations: int
    @property
    def bifurcated(self): return self.solution_type == 'nontrivial'

class CDStateUpdater:
    def __init__(self, d_state=D_STATE, L=1.0, c=10.0, beta_scale=1.2):
        self.d_state=d_state; self.N=d_state; self.L=L; self.c=c
        self.beta_scale=beta_scale; self.records=[]
        if CD_AVAILABLE:
            self._beta_star_unit = viability_threshold_1d(L, b=1.0)

    def _message_to_viability(self, msg):
        m = msg.flatten()[:self.N]
        lo,hi = m.min(),m.max()
        b = np.ones(self.N)*0.5 if hi-lo<1e-8 else (m-lo)/(hi-lo)
        return b, float(b.mean())

    def _query_to_drive(self, q):
        q = q.flatten()[:D_QUERY]
        a = np.interp(np.linspace(0,1,self.N), np.linspace(0,1,len(q)), q)
        return np.abs(a)

    def update(self, compressed, state, goal_query, step=0, module_id='?'):
        msg_np   = compressed.detach().numpy().flatten()
        query_np = goal_query.detach().numpy().flatten()
        b_field, b_mean = self._message_to_viability(msg_np)
        a_field = self._query_to_drive(query_np)
        beta_star = viability_threshold_1d(self.L, b=b_mean) if b_mean>1e-8 else float(self._beta_star_unit)
        beta_b = self.beta_scale * beta_star
        try:
            x, Phi, info = solve_1d_picard(L=self.L, N=self.N, a=float(a_field.mean()), beta_b=beta_b, c=self.c)
            converged = check_convergence(info)[0]
            sol_type  = solution_type(info)
            max_phi   = float(info.get('maxPhi', np.abs(Phi).max()))
            n_iter    = int(info.get('iterations', -1))
            # Always clamp to exactly N points
            Phi = Phi.flatten()[:self.N] if len(Phi) > self.N else Phi.flatten()
            Phi = Phi[:self.N]  # hard clamp
            state_norm = float(torch.norm(state).item())
            phi_norm   = float(np.linalg.norm(Phi))
            if phi_norm > 1e-8 and state_norm > 1e-8:
                Phi = Phi * (state_norm / phi_norm)
            new_state = torch.tensor(Phi, dtype=torch.float32).unsqueeze(0)
        except Exception as e:
            warnings.warn(f"CD solver failed: {e}. Using current state.")
            new_state = state; converged=False; sol_type='trivial'; max_phi=0.0; beta_star=0.0; n_iter=0
        self.records.append(CDUpdateRecord(step=step, module_id=module_id, converged=converged,
            solution_type=sol_type, max_phi=max_phi, beta_used=beta_b,
            beta_star=beta_star, above_threshold=beta_b>beta_star, iterations=n_iter))
        return new_state

    def summary(self):
        if not self.records: return {}
        n = len(self.records)
        return {'total_updates':n, 'converged':sum(1 for r in self.records if r.converged),
                'convergence_rate':sum(1 for r in self.records if r.converged)/n,
                'nontrivial':sum(1 for r in self.records if r.bifurcated),
                'bifurcation_rate':sum(1 for r in self.records if r.bifurcated)/n,
                'mean_max_phi':np.mean([r.max_phi for r in self.records])}

class FabricStateUpdater(nn.Module):
    def __init__(self, d_msg=D_MSG, d_state=D_STATE, d_query=D_QUERY, use_cd=False, beta_scale=1.2):
        super().__init__()
        self.use_cd=use_cd; self.d_state=d_state
        self.gru = nn.GRUCell(d_msg, d_state)
        self.cd  = CDStateUpdater(d_state=d_state, beta_scale=beta_scale) if CD_AVAILABLE else None
        if use_cd and not CD_AVAILABLE:
            warnings.warn("use_cd=True but CD not installed. Falling back to GRU.")
            self.use_cd=False

    def forward(self, compressed, state, goal_query, step=0, module_id='?'):
        if self.use_cd and self.cd is not None:
            return self.cd.update(compressed, state, goal_query, step, module_id)
        return self.gru(compressed, state)

    def cd_summary(self): return self.cd.summary() if self.cd else {}
    def switch_to_cd(self):
        if not CD_AVAILABLE: raise RuntimeError("CD not installed.")
        self.use_cd=True
    def switch_to_gru(self): self.use_cd=False

Overwriting /mnt/custom-file-systems/s3/shared/ogi_navi_sad/cd_state_updater.py
